# LNUQNet – Inference Only for Plots and Uncertainty Quantification

In [ ]:
# =========================================================
# LNUQNet – Libraries Used
# =========================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

# =========================================================
# Reproducibility
# =========================================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# =========================================================
# Configuration
# =========================================================

WINDOW_SIZE = 30
BATCH_SIZE = 32

DEPTH_COL = "Depth.m"
LABEL_COL = "Lithology"

CSV_FILES = [
    "classified_A10.csv", "classified_A15.csv", "classified_A16.csv",
    "classified_B2.csv", "classified_B4.csv", "classified_B8.csv", "classified_B9.csv",
    "classified_C1.csv", "classified_C2.csv",
    "classified_C3.csv", "classified_C4.csv", "classified_C5.csv"
]

TRAIN_WELLS = [
    'classified_C2', 'classified_C4', 'classified_C3',
    'classified_B4', 'classified_A15', 'classified_A16',
    'classified_A10', 'classified_B2', 'classified_B8'
]

TEST_WELLS = [
    'classified_C5', 'classified_B9', 'classified_C1'
]

# =========================================================
# Data Loading & Preparation
# =========================================================

def load_well_csvs(csv_files):
    dfs = []
    for path in csv_files:
        df = pd.read_csv(path)
        well_name = os.path.basename(path).replace(".csv", "")
        df["WELL"] = well_name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def sliding_window_generator(df, feature_cols, label_col, window):
    X, y = [], []
    labels = df[label_col].values

    for i in range(len(df) - window + 1):
        window_df = df.iloc[i:i + window]

        # Majority vote label
        label_window = labels[i:i + window]
        label = np.bincount(label_window).argmax()

        X.append(window_df[feature_cols].values)
        y.append(label)

    return np.array(X), np.array(y)


def prepare_data(df, window_size):
    feature_cols = [c for c in df.columns if c not in [LABEL_COL, "WELL"]]

    X_all, y_all, wells_all = [], [], []

    for well in df["WELL"].unique():
        df_well = df[df["WELL"] == well].reset_index(drop=True)

        X_w, y_w = sliding_window_generator(
            df_well,
            feature_cols,
            LABEL_COL,
            window_size
        )

        X_all.append(X_w)
        y_all.append(y_w)
        wells_all.extend([well] * len(y_w))

    X = np.vstack(X_all)
    y = np.hstack(y_all)
    wells = np.array(wells_all)

    return X, y, wells, feature_cols

# =========================================================
# Model Components
# =========================================================

def transformer_encoder(inputs, d_model=128, num_heads=4, dff=256, dropout=0.1):
    x = layers.Dense(d_model)(inputs)

    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(dff, activation="relu")(x)
    ffn = layers.Dense(d_model)(ffn)

    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    return x

# =========================================================
# Main Execution (Inference Only)
# =========================================================

if __name__ == "__main__":

    print("\n Loading data...")
    df = load_well_csvs(CSV_FILES)

    print("Encoding labels...")
    le = LabelEncoder()
    df[LABEL_COL] = le.fit_transform(df[LABEL_COL])

    print("Creating sliding windows...")
    X, y, wells, feature_cols = prepare_data(df, WINDOW_SIZE)

    print("Normalizing features (reconstructed scaler)...")
    scaler = StandardScaler()
    X = scaler.fit_transform(
        X.reshape(-1, X.shape[-1])
    ).reshape(X.shape)

    # Add CNN channel
    X = X[..., np.newaxis]

    print(" Applying well-based split...")
    train_mask = np.isin(wells, TRAIN_WELLS)
    test_mask  = np.isin(wells, TEST_WELLS)

    X_train_full, y_train_full = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    assert len(X_test) > 0, "Test set is empty!"

    # =====================================================
    # Load trained model (NO COMPILE, NO FIT)
    # =====================================================

    print("\n Loading trained LNUQNet model...")
    model = tf.keras.models.load_model(
        "best_lnuqnet_model.keras",
        custom_objects={
            "transformer_encoder": transformer_encoder
        }
    )

    model.summary()

# Utility Functions

# Classification Report

In [ ]:
print("\n Running inference on train wells...")
probs_train = model.predict(X_train_full, batch_size=BATCH_SIZE)
y_pred_train = np.argmax(probs_train, axis=1)

print("\n Classification Report (TRAIN WELLS)")
print(classification_report(
    y_train_full,
    y_pred_train,
    target_names=[str(c) for c in le.classes_]
))

In [ ]:
# =====================================================
# Evaluation
# =====================================================

print("\n🔹 Running inference on test wells...")
probs = model.predict(X_test, batch_size=BATCH_SIZE)
y_pred = np.argmax(probs, axis=1)

print("\n📊 Classification Report (TEST WELLS)")
print(classification_report(
  y_test,
  y_pred,
  target_names=[str(c) for c in le.classes_]
))

## Per Well Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

print("\n=== Per-Well Train Accuracy ===")

for well in np.unique(wells[train_mask]):
    idx = np.where((wells == well) & train_mask)[0]

    X_well = X[idx]
    y_well = y[idx]

    y_pred = np.argmax(model.predict(X_well, verbose=0), axis=1)

    acc = accuracy_score(y_well, y_pred)

    print(f"{well:25s} | Accuracy: {acc:.3f} | Samples: {len(y_well)}")


In [ ]:
from sklearn.metrics import accuracy_score

print("\n=== Per-Well Test Accuracy ===")

for well in np.unique(wells[test_mask]):
    idx = np.where((wells == well) & test_mask)[0]

    X_well = X[idx]
    y_well = y[idx]

    y_pred = np.argmax(model.predict(X_well, verbose=0), axis=1)

    acc = accuracy_score(y_well, y_pred)

    print(f"{well:25s} | Accuracy: {acc:.3f} | Samples: {len(y_well)}")


# Well log Plots


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

# =========================================================
# LITHOLOGY SETTINGS
# =========================================================
LITHO_COLORS = [
    "#E6E6E6",  # Shale
    "#D2D296",  # Shaly Sandstone
    "#FFFF00",  # Sandstone
]

LITHO_LABELS = [
    "Shale",
    "Shaly Sandstone",
    "Sandstone",
]

LITHO_HATCHES = {
    0: '----',
    1: ['..', '--'],
    2: '....',
}

litho_cmap = ListedColormap(LITHO_COLORS)

# =========================================================
# HELPERS
# =========================================================
def get_intervals(depths, labels):
    if len(depths) == 0:
        return []

    intervals = []
    start_z = depths[0]
    curr_label = labels[0]

    for z, l in zip(depths[1:], labels[1:]):
        if l != curr_label:
            intervals.append((start_z, z, curr_label))
            curr_label = l
            start_z = z

    intervals.append((start_z, depths[-1], curr_label))
    return intervals


def add_lithology_patterns(ax, intervals, x_pos=0, width=1):
    for top, base, lith_code in intervals:
        height = base - top
        pattern = LITHO_HATCHES.get(int(lith_code))

        if pattern:
            patterns = pattern if isinstance(pattern, list) else [pattern]
            for p in patterns:
                ax.add_patch(
                    mpatches.Rectangle(
                        (x_pos, top), width, height,
                        facecolor='none',
                        edgecolor='black',
                        linewidth=0,
                        hatch=p,
                        alpha=0.35
                    )
                )

# =========================================================
# MAIN FUNCTION
# =========================================================
def plot_well_petrel_style(
    df_well,
    model,
    scaler,
    feature_cols,
    label_col,
    window_size,
    well_name,
    DEPTH_COL="Depth.m",
    save_dir="well_plots"
):

    os.makedirs(save_dir, exist_ok=True)

    depth = df_well[DEPTH_COL].values
    true_lith = df_well[label_col].values
    n_depth = len(depth)

    plot_track_cols = [
        c for c in df_well.columns
        if c not in [label_col, "WELL", DEPTH_COL]
    ]

    # -----------------------------------------------------
    # Sliding window prediction
    # -----------------------------------------------------
    X_well, center_idx = [], []

    for i in range(n_depth - window_size + 1):
        window_df = df_well.iloc[i:i + window_size]
        X_well.append(window_df[feature_cols].values)
        center_idx.append(i + window_size // 2)

    X_well = np.array(X_well)
    center_idx = np.array(center_idx)

    X_well = scaler.transform(
        X_well.reshape(-1, X_well.shape[-1])
    ).reshape(X_well.shape)

    X_well = X_well[..., np.newaxis]
    preds = np.argmax(model.predict(X_well, verbose=0), axis=1)

    pred_lith = np.full(n_depth, np.nan)
    pred_lith[center_idx] = preds

    valid_mask = ~np.isnan(pred_lith)
    first_valid = np.where(valid_mask)[0][0]
    last_valid = np.where(valid_mask)[0][-1] + 1

    depth = depth[first_valid:last_valid]
    true_lith = true_lith[first_valid:last_valid]
    pred_lith = pred_lith[first_valid:last_valid]
    df_well = df_well.iloc[first_valid:last_valid].reset_index(drop=True)

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------
    depth_range = depth.max() - depth.min()
    fig_height = max(8, depth_range / 40)

    n_tracks = len(plot_track_cols) + 2

    fig = plt.figure(figsize=(2.3 * n_tracks, fig_height + 2.2))
    gs = fig.add_gridspec(2, n_tracks, height_ratios=[fig_height, 0.7])

    axes = [fig.add_subplot(gs[0, i]) for i in range(n_tracks)]
    n_features = len(plot_track_cols)
    legend_ax = fig.add_subplot(gs[1, :n_features])


    # -----------------------------------------------------
    # Depth axis
    # -----------------------------------------------------
    for i, ax in enumerate(axes):
        if i == 0:
            ax.set_ylabel("Depth (m)", fontsize=12, fontweight="bold")
        else:
            ax.sharey(axes[0])
            ax.tick_params(axis='y', left=False, labelleft=False)

    # -----------------------------------------------------
    # Logs
    # -----------------------------------------------------
    for i, col in enumerate(plot_track_cols):
        ax = axes[i]
        ax.plot(df_well[col], depth, color="black", linewidth=1.5)
        ax.set_xlabel(col, fontsize=11, fontweight="bold")
        ax.invert_yaxis()
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.xaxis.set_major_locator(MaxNLocator(3))
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.4g'))

    # -----------------------------------------------------
    # True lithology
    # -----------------------------------------------------
    ax_true = axes[-2]
    ax_true.imshow(
        true_lith.reshape(-1, 1),
        cmap=litho_cmap,
        aspect="auto",
        origin="upper",
        extent=[0, 1, depth.max(), depth.min()]
    )
    add_lithology_patterns(ax_true, get_intervals(depth, true_lith))
    ax_true.set_title("Actual", fontsize=12, fontweight="bold")
    ax_true.set_xticks([])

    # -----------------------------------------------------
    # Predicted lithology
    # -----------------------------------------------------
    ax_pred = axes[-1]
    ax_pred.imshow(
        pred_lith.reshape(-1, 1),
        cmap=litho_cmap,
        aspect="auto",
        origin="upper",
        extent=[0, 1, depth.max(), depth.min()]
    )
    add_lithology_patterns(ax_pred, get_intervals(depth, pred_lith))
    ax_pred.set_title("LNUQNet", fontsize=12, fontweight="bold")
    ax_pred.set_xticks([])

    # =====================================================
    # LEGEND — LABELS UNDER BOXES
    # =====================================================
    legend_ax.axis("off")
    legend_ax.set_xlim(0, 1)
    legend_ax.set_ylim(0, 1)

    num_lithos = len(LITHO_LABELS)

    box_width = 0.25
    box_height = 0.55
    y_box = 0.35
    y_text = 0.12

    x_positions = np.linspace(0.15, 0.85, num_lithos)

    for i, (x, color, label) in enumerate(zip(x_positions, LITHO_COLORS, LITHO_LABELS)):
        # Base box
        legend_ax.add_patch(
            mpatches.Rectangle(
                (x - box_width / 2, y_box),
                box_width, box_height,
                facecolor=color,
                edgecolor="black",
                linewidth=1.3
            )
        )

        # Hatch overlay
        pattern = LITHO_HATCHES.get(i)
        if pattern:
            patterns = pattern if isinstance(pattern, list) else [pattern]
            for p in patterns:
                legend_ax.add_patch(
                    mpatches.Rectangle(
                        (x - box_width / 2, y_box),
                        box_width, box_height,
                        facecolor='none',
                        edgecolor='black',
                        linewidth=0,
                        hatch=p,
                        alpha=0.6
                    )
                )

        # Text UNDER the box
        legend_ax.text(
            x, y_text,
            label,
            ha="center",
            va="top",
            fontsize=13,
            fontweight="bold"
        )

    # -----------------------------------------------------
    # Title and save
    # -----------------------------------------------------
    display_name = well_name.replace("classified_", "")
    fig.suptitle(
        f"{display_name} | {depth.min():.1f}–{depth.max():.1f} m",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.99])

    out_path = os.path.join(save_dir, f"{well_name}_petrel_style.png")
    plt.savefig(out_path, dpi=650, bbox_inches="tight")
    # plt.show()


## Train LOG Track

In [ ]:
for well in TRAIN_WELLS:
    df_w = df[df["WELL"] == well].reset_index(drop=True)

    plot_well_petrel_style(
        df_well=df_w,
        model=model,
        scaler=scaler,
        feature_cols=feature_cols,
        label_col=LABEL_COL,
        window_size=WINDOW_SIZE,
        well_name=well
    )


## Test LOG Track

In [ ]:
for well in TEST_WELLS:
    df_w = df[df["WELL"] == well].reset_index(drop=True)

    plot_well_petrel_style(
        df_well=df_w,
        model=model,
        scaler=scaler,
        feature_cols=feature_cols,
        label_col=LABEL_COL,
        window_size=WINDOW_SIZE,
        well_name=well
    )


# Monte Carlo Uncertainty Quantification (MC-UQ)

In [ ]:
import numpy as np
import tensorflow as tf

# =========================================================
# Monte Carlo Dropout Prediction
# =========================================================
def mc_dropout_predict(model, X, n_mc=100, batch_size=256):
    """
    Perform Monte Carlo forward passes with dropout enabled.

    Parameters
    ----------
    model : tf.keras.Model
        Trained LNUQNet model with dropout layers
    X : np.ndarray
        Preprocessed input data
    n_mc : int
        Number of Monte Carlo samples
    batch_size : int
        Batch size for inference

    Returns
    -------
    np.ndarray
        Shape (n_mc, n_samples, n_classes)
    """
    mc_preds = []

    for i in range(n_mc):
        preds = model(X, training=True)
        mc_preds.append(preds.numpy())

    return np.array(mc_preds)


# =========================================================
# Uncertainty Metrics
# =========================================================
def predictive_mean(mc_preds):
    """
    Mean predictive probability.
    """
    return np.mean(mc_preds, axis=0)


def predictive_variance(mc_preds):
    """
    Variance of predictive probabilities.
    """
    return np.var(mc_preds, axis=0)


def predictive_entropy(mean_probs, eps=1e-8):
    """
    Entropy of the mean predictive distribution.
    """
    return -np.sum(mean_probs * np.log(mean_probs + eps), axis=1)


def expected_entropy(mc_preds, eps=1e-8):
    """
    Expected entropy across Monte Carlo samples.
    """
    entropies = -np.sum(mc_preds * np.log(mc_preds + eps), axis=2)
    return np.mean(entropies, axis=0)


def mutual_information(mc_preds, eps=1e-8):
    """
    Mutual information capturing epistemic uncertainty.
    """
    mean_probs = predictive_mean(mc_preds)
    pe = predictive_entropy(mean_probs, eps)
    ee = expected_entropy(mc_preds, eps)
    return pe - ee


# =========================================================
# Full Uncertainty Pipeline
# =========================================================
def compute_mc_uncertainty(model, X, n_mc=100):
    """
    Compute all Monte Carlo uncertainty metrics.

    Returns
    -------
    dict
        Dictionary containing uncertainty metrics
    """
    mc_preds = mc_dropout_predict(model, X, n_mc=n_mc)

    mean_probs = predictive_mean(mc_preds)
    var_probs = predictive_variance(mc_preds)
    entropy = predictive_entropy(mean_probs)
    mi = mutual_information(mc_preds)

    results = {
        "mc_predictions": mc_preds,
        "predictive_mean": mean_probs,
        "predictive_variance": var_probs,
        "predictive_entropy": entropy,
        "mutual_information": mi,
        "predicted_class": np.argmax(mean_probs, axis=1),
        "confidence": np.max(mean_probs, axis=1)
    }

    return results

In [ ]:
# =========================================================
# Run Monte Carlo Uncertainty
# =========================================================
N_MC = 100

results = compute_mc_uncertainty(
    model=model,
    X=X,                   # preprocessed input
    n_mc=N_MC
)

# =========================================================
# Explicit Outputs
# =========================================================
print("\nFirst 5 predicted classes:", results["predicted_class"][:5])
print("First 5 prediction confidences:", results["confidence"][:5])
print("First 5 predictive entropies:", results["predictive_entropy"][:5])
print("First 5 mutual information values:", results["mutual_information"][:5])

## MC-UQ Report

In [ ]:
import pandas as pd

# Build analysis dataframe
df_uq = pd.DataFrame({
    "pred_class": results["predicted_class"],
    "confidence": results["confidence"],
    "entropy": results["predictive_entropy"],
    "mutual_information": results["mutual_information"]
})

# Overall statistics
summary = df_uq.describe()
print(summary)

# Stratify by confidence
high_conf = df_uq[df_uq["confidence"] > 0.9]
low_conf = df_uq[df_uq["confidence"] <= 0.9]

print("\nHigh confidence entropy mean:", high_conf["entropy"].mean())
print("Low confidence entropy mean:", low_conf["entropy"].mean())

# Stratify by class
class_stats = df_uq.groupby("pred_class").agg({
    "entropy": ["mean", "median"],
    "mutual_information": ["mean", "median"],
    "confidence": ["mean"]
})

print("\nUncertainty by predicted class:")
print(class_stats)

## MC-UQ Log Tracks

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# =========================================================
# LITHOLOGY SETTINGS
# =========================================================
LITHO_COLORS = [
    "#E6E6E6",  # Shale
    "#D2D296",  # Shaly Sandstone
    "#FFFF00",  # Sandstone
]

LITHO_LABELS = [
    "Shale",
    "Shaly Sandstone",
    "Sandstone",
]

LITHO_HATCHES = {
    0: '----',
    1: ['..', '--'],
    2: '....',
}

litho_cmap = ListedColormap(LITHO_COLORS)

# =========================================================
# HELPERS
# =========================================================
def get_intervals(depths, labels):
    if len(depths) == 0:
        return []

    intervals = []
    start_z = depths[0]
    curr_label = labels[0]

    for z, l in zip(depths[1:], labels[1:]):
        if l != curr_label:
            intervals.append((start_z, z, curr_label))
            curr_label = l
            start_z = z

    intervals.append((start_z, depths[-1], curr_label))
    return intervals

def add_lithology_patterns(ax, intervals, x_pos=0, width=1):
    for top, base, lith_code in intervals:
        height = base - top
        pattern = LITHO_HATCHES.get(int(lith_code))
        if pattern:
            patterns = pattern if isinstance(pattern, list) else [pattern]
            for p in patterns:
                ax.add_patch(
                    mpatches.Rectangle(
                        (x_pos, top),
                        width,
                        height,
                        facecolor='none',
                        edgecolor='black',
                        linewidth=0,
                        hatch=p,
                        alpha=0.35
                    )
                )

# =========================================================
# MAIN DEPTH-UNCERTAINTY PLOT
# =========================================================
def plot_depth_uncertainty(df_well, model, scaler, feature_cols, label_col,
                           window_size, well_name, DEPTH_COL="Depth.m", N_MC=100, save_dir="well_plots_uq"):

    os.makedirs(save_dir, exist_ok=True)

    depth = df_well[DEPTH_COL].values
    true_lith = df_well[label_col].values
    n_depth = len(depth)

    # -----------------------------------------------------
    # Sliding window preparation
    # -----------------------------------------------------
    X_well, center_idx = [], []
    for i in range(n_depth - window_size + 1):
        window_df = df_well.iloc[i:i + window_size]
        X_well.append(window_df[feature_cols].values)
        center_idx.append(i + window_size // 2)

    X_well = np.array(X_well)
    center_idx = np.array(center_idx)
    X_well = scaler.transform(X_well.reshape(-1, X_well.shape[-1])).reshape(X_well.shape)
    X_well = X_well[..., np.newaxis]

    # -----------------------------------------------------
    # Monte Carlo Dropout Inference
    # -----------------------------------------------------
    mc_preds = [model(X_well, training=True).numpy() for _ in range(N_MC)]
    mc_preds = np.array(mc_preds)

    mean_probs = np.mean(mc_preds, axis=0)
    preds = np.argmax(mean_probs, axis=1)
    confidence = np.max(mean_probs, axis=1)

    # -----------------------------------------------------
    # Align predictions and confidence to depth
    # -----------------------------------------------------
    pred_lith = np.full(n_depth, np.nan)
    conf_full = np.full(n_depth, np.nan)
    pred_lith[center_idx] = preds
    conf_full[center_idx] = confidence

    valid_mask = ~np.isnan(pred_lith)
    first_valid = np.where(valid_mask)[0][0]
    last_valid = np.where(valid_mask)[0][-1] + 1

    depth = depth[first_valid:last_valid]
    true_lith = true_lith[first_valid:last_valid]
    pred_lith = pred_lith[first_valid:last_valid]
    conf_full = conf_full[first_valid:last_valid]

    # -----------------------------------------------------
    # Plot
    # -----------------------------------------------------
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(6, 12), sharey=True)

    # True lithology
    ax_true = axes[0]
    ax_true.imshow(
        true_lith.reshape(-1, 1),
        cmap=litho_cmap,
        aspect="auto",
        origin="upper",
        extent=[0, 1, depth.max(), depth.min()]
    )
    add_lithology_patterns(ax_true, get_intervals(depth, true_lith))
    ax_true.set_title("Actual", fontsize=12, fontweight="bold")
    ax_true.set_xticks([])

    # Predicted lithology
    ax_pred = axes[1]
    ax_pred.imshow(
        pred_lith.reshape(-1, 1),
        cmap=litho_cmap,
        aspect="auto",
        origin="upper",
        extent=[0, 1, depth.max(), depth.min()]
    )
    add_lithology_patterns(ax_pred, get_intervals(depth, pred_lith))
    ax_pred.set_title("LNUQNet", fontsize=12, fontweight="bold")
    ax_pred.set_xticks([])

    # Confidence
    ax_conf = axes[2]
    ax_conf.plot(conf_full, depth, color="black", linewidth=1.5)
    ax_conf.set_title("MC Confidence", fontsize=12, fontweight="bold")
    ax_conf.set_xlim(0, 1)
    ax_conf.grid(True, linestyle="--", alpha=0.4)

    axes[0].set_ylabel("Depth (m)", fontsize=12, fontweight="bold")
    axes[0].invert_yaxis()

    display_name = well_name.replace("classified_", "")
    fig.suptitle(
        f"{display_name} | {depth.min():.1f}–{depth.max():.1f} m",
        fontsize=12,
        fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])

    out_path = os.path.join(save_dir, f"{well_name}_depth_uncertainty.png")
    plt.savefig(out_path, dpi=650, bbox_inches="tight")

### MC-UQ Log Tracks for Train Set

In [ ]:
print("\n=== Generating MC UQ plots for TRAIN WELLS ===")
for well in TRAIN_WELLS:
    df_w = df[df["WELL"] == well].reset_index(drop=True)

    plot_depth_uncertainty(
        df_well=df_w,
        model=model,
        scaler=scaler,
        feature_cols=feature_cols,
        label_col=LABEL_COL,
        window_size=WINDOW_SIZE,
        well_name=well,
        DEPTH_COL="Depth.m",
        N_MC=100,
        save_dir="well_plots_uq"
    )
print("Finished generating MC UQ plots for TRAIN WELLS.")

### MC-UQ Log Tracks for Test Set

In [ ]:
print("\n=== Generating MC UQ plots for TEST WELLS ===")
for well in TEST_WELLS:
    df_w = df[df["WELL"] == well].reset_index(drop=True)

    plot_depth_uncertainty(
        df_well=df_w,
        model=model,
        scaler=scaler,
        feature_cols=feature_cols,
        label_col=LABEL_COL,
        window_size=WINDOW_SIZE,
        well_name=well,
        DEPTH_COL="Depth.m",
        N_MC=100,
        save_dir="well_plots_uq"
    )
print("Finished generating MC UQ plots for TEST WELLS.")